# Most recent benchmarks for BRCA dataset

In [2]:
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import polars as pl
import torch
import torch_geometric as pyg
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split, StratifiedKFold

from bipartite_gnn.graph_visualizatons import visualize_graph, visualize_embeddings
from baseline_evals.feature_selection import variance_filtering, class_variational_selection
from bipartite_gnn.preprocessing import mrmr_selection
from baseline_evals.knn_eval import knn_eval
from baseline_evals.svm_eval import svm_eval
from baseline_evals.xgboost_eval import xgboost_eval
from baseline_evals.mlp_eval import mlp_eval
from bipartite_gnn.graph_building import cosine_similarity_matrix, keep_n_neighbours, dense_to_coo
from bipartite_gnn.train_test import GNNTrainer
from gnn_experiments.mogonet_eval import mogonet_eval

### Load data and make sure that everything is aligned

In [3]:
null_vals = ["NA"]

mrna = pl.read_csv("BRCA_PROCESSED_DATA/mrna.tsv", separator="\t", null_values=null_vals)
meth = pl.read_csv("BRCA_PROCESSED_DATA/meth.tsv", separator="\t", null_values=null_vals)
mirna = pl.read_csv("BRCA_PROCESSED_DATA/mirna.tsv", separator="\t", null_values=null_vals)
cnv = pl.read_csv("BRCA_PROCESSED_DATA/cnvth.tsv", separator="\t", null_values=null_vals)

labels = pl.read_csv("BRCA_PROCESSED_DATA/labels.tsv", separator="\t")
le = LabelEncoder()
le.fit(labels["PAM50_mRNA_nature2012"].to_list())
y = le.transform(labels["PAM50_mRNA_nature2012"].to_list())

assert mrna.columns[1:] == meth.columns[1:] == cnv.columns[1:] == mirna.columns[1:] == labels["sampleID"].to_list()

In [3]:
# concat all data into one dataframe
mrna_X = mrna[:, 1:].to_numpy().T
meth_X = meth[:, 1:].to_numpy().T
mirna_X = mirna[:, 1:].to_numpy().T
cnv_X = cnv[:, 1:].to_numpy().T

print(mrna_X.shape, meth_X.shape, mirna_X.shape, cnv_X.shape)

# concat after preprocessing
X_list = [mrna_X, meth_X, mirna_X]

# concat before preprocessing
X = np.hstack(X_list)

(483, 18975) (483, 9317) (483, 231) (483, 24776)


In [82]:
knn_eval(X, y, select_n_features=True)

500
| KNN | 0.79 +/- 0.02 | 0.78 +/- 0.03 | 0.77 +/- 0.03 |
study.best_value=0.7731821746220054, study.best_params={'n_neighbors': 6, 'n_features': 3755}


## concat before

| Model | ACC | F1 macro | F1 weighted |
| --- | --- | --- | --- | 
| KNN (mrna) | 0.84 +/- 0.02 | 0.83 +/- 0.02 | 0.83 +/- 0.02 |
| KNN (mrna, mirna) | 0.84 +/- 0.02 | 0.83 +/- 0.03 | 0.84 +/- 0.02 |
| KNN (mrna, mirna, meth) | 0.80 +/- 0.04 | 0.78 +/- 0.06 | 0.79 +/- 0.05 |
| KNN (mrna, mirna, meth, cnv) | 0.80 +/- 0.04 | 0.78 +/- 0.05 | 0.79 +/- 0.05 |

## concat after (KNN)

| subset | weighted f1 |
| --- | --- |
| mrna | 0.83 +/- 0.01 |
| mirna | 0.68 +/- 0.04 |
| meth | 0.61 +/- 0.07 |
| cnv | 0.55 +/- 0.03 |
| mrna, mirna | 0.82 +/- 0.03 |
| mrna, mirna, meth | 0.78 +/- 0.03 |
| mrna, mirna, meth, cnv | 0.72 +/- 0.03 |

In [83]:
svm_eval(X, y, select_n_features=True, n_trials=20, n_features_preselect=10000)

Trial 1 / 20
| SVM | 0.83 +/- 0.01 | 0.83 +/- 0.02 | 0.83 +/- 0.01 |
Trial 2 / 20
Trial 3 / 20
Pruning trial
Trial 4 / 20
Pruning trial
Trial 5 / 20


KeyboardInterrupt: 

In [ ]:
| LIN SVM (mrna) | 0.83 +/- 0.01 | 0.83 +/- 0.01 | 0.83 +/- 0.01 |
| LIN SVM (all) | 0.83 +/- 0.01 | 0.83 +/- 0.02 | 0.83 +/- 0.01 |

In [84]:
xgboost_eval(X, y, n_trials=20)

0 / 20
ACC: [0.81443299 0.75257732 0.77319588 0.79166667 0.79166667]
F1M: [0.75649166 0.7171824  0.73991438 0.78683275 0.77552635]
F1W: [0.81636058 0.74620681 0.76904404 0.79061621 0.79111649]
 XGBoost val | 0.78 +/- 0.02 | 0.72 +/- 0.05 | 0.78 +/- 0.03 |
| XGBoost | 0.78 +/- 0.02 | 0.76 +/- 0.02 | 0.78 +/- 0.02 |
1 / 20
ACC: [0.75257732 0.8556701  0.7628866  0.79166667 0.78125   ]
F1M: [0.73299234 0.86114947 0.76676267 0.76812008 0.7829877 ]
F1W: [0.75819309 0.85681546 0.7537579  0.78380265 0.78443131]
 XGBoost val | 0.79 +/- 0.02 | 0.76 +/- 0.05 | 0.79 +/- 0.03 |
| XGBoost | 0.79 +/- 0.04 | 0.78 +/- 0.04 | 0.79 +/- 0.04 |
2 / 20
ACC: [0.8556701  0.8556701  0.86597938 0.875      0.8125    ]
F1M: [0.83637566 0.85521223 0.85638371 0.87290121 0.8058126 ]
F1W: [0.85863198 0.85355805 0.86391041 0.87154538 0.81443624]
 XGBoost val | 0.85 +/- 0.05 | 0.84 +/- 0.03 | 0.85 +/- 0.05 |
| XGBoost | 0.85 +/- 0.02 | 0.85 +/- 0.02 | 0.85 +/- 0.02 |
3 / 20
Pruning trial
4 / 20
ACC: [0.81443299 0.89690

KeyboardInterrupt: 

| XGBoost (default) | 0.80 +/- 0.02 | 0.80 +/- 0.04 | 0.80 +/- 0.03 |  
| XGBoost (tuned, mrna only) | 0.88 +/- 0.02 | 0.88 +/- 0.03 | 0.88 +/- 0.02 |  
| XGBoost (tuned, all) | 0.88 +/- 0.03 | 0.87 +/- 0.04 | 0.88 +/- 0.03 |  

In [85]:
mlp_eval(X, y, reg_type=None, n_trials=15)

Trial 0 / 15
Eval 1 / 5
Eval 2 / 5
Eval 3 / 5
Eval 4 / 5
Eval 5 / 5
[0.79591835 0.83673471 0.77551019 0.85416669 0.79166669]
[0.77526596 0.85965198 0.78898936 0.86764706 0.77646025]
[0.79027356 0.84042536 0.77389926 0.84272876 0.79224308]
{'acc': 0.8107993245124817, 'f1_macro': 0.8136029217249272, 'f1_weighted': 0.8079140029225609, 'acc_std': 0.0296182263899997, 'f1_macro_std': 0.04122235322546163, 'f1_weighted_std': 0.028223468685685985}
Trial 1 / 15
Eval 1 / 5
Eval 2 / 5
Eval 3 / 5
Eval 4 / 5
Eval 5 / 5
[0.79591835 0.89795917 0.73469388 0.85416669 0.79166669]
[0.75640732 0.90118577 0.72564103 0.86150794 0.77787264]
[0.78696119 0.89658788 0.69911041 0.84794974 0.79150065]
Trial 2 / 15
Eval 1 / 5
Eval 2 / 5
Eval 3 / 5
Pruning trial after 3 evals, cause 0.32279998122428116 < 0.8079140029225609
[0.2857143  0.44897959 0.40816328 0.         0.        ]
[0.20712582 0.26785714 0.22222222 0.         0.        ]
[0.26659122 0.37900875 0.33786848 0.         0.        ]
Trial 3 / 15
Eval 1 / 5
E

| MLP no reg | 0.86 +/- 0.06 | 0.83 +/- 0.09 | 0.86 +/- 0.05 |  
| MLP no reg (mRNA only) | 0.87 +/- 0.04 | 0.85 +/- 0.05 | 0.87 +/- 0.04 |  
| MLP no reg (mRNA only) | 0.87 +/- 0.04 | 0.87 +/- 0.06 | 0.87 +/- 0.04 | 

In [86]:
mlp_eval(X, y, reg_type="l1", n_trials=15)

Trial 0 / 15
Eval 1 / 5
torch.Size([386, 1356])
torch.Size([48, 1356])
torch.Size([49, 1356])
Eval 2 / 5
torch.Size([386, 1356])
torch.Size([48, 1356])
torch.Size([49, 1356])
Eval 3 / 5
torch.Size([386, 1356])
torch.Size([48, 1356])
torch.Size([49, 1356])
Eval 4 / 5
torch.Size([387, 1356])
torch.Size([48, 1356])
torch.Size([48, 1356])
Eval 5 / 5
torch.Size([387, 1356])
torch.Size([48, 1356])
torch.Size([48, 1356])
[0.46938777 0.71428573 0.65306121 0.60416669 0.52083331]
[0.47993159 0.75023034 0.67177889 0.54740575 0.52323718]
[0.47600338 0.71219225 0.64396686 0.58800703 0.53427707]
{'acc': 0.5923469424247741, 'f1_macro': 0.5945167513955909, 'f1_weighted': 0.5908893179886208, 'acc_std': 0.08823622191961213, 'f1_macro_std': 0.10064406970572103, 'f1_weighted_std': 0.08239376048341274}
Trial 1 / 15
Eval 1 / 5
torch.Size([386, 1506])
torch.Size([48, 1506])
torch.Size([49, 1506])
Eval 2 / 5
torch.Size([386, 1506])
torch.Size([48, 1506])
torch.Size([49, 1506])
Eval 3 / 5
torch.Size([386, 1506

Exception ignored in: <function _releaseLock at 0x77cf631bcea0>
Traceback (most recent call last):
  File "/home/lubojjan/micromamba/envs/diploma/lib/python3.12/logging/__init__.py", line 243, in _releaseLock
    def _releaseLock():
    
KeyboardInterrupt: 


Eval 3 / 5
torch.Size([386, 2523])
torch.Size([48, 2523])
torch.Size([49, 2523])


| MLP l1 reg | 0.87 +/- 0.06 | 0.85 +/- 0.09 | 0.87 +/- 0.05 |  
| MLP l1 reg (mrna only) | 0.86 +/- 0.03 | 0.85 +/- 0.04 | 0.86 +/- 0.03 |  
| MLP l1 reg (mrna only) | 0.86 +/- 0.07 | 0.86 +/- 0.08 | 0.86 +/- 0.07 |  
| MLP l1 reg (mrna only, var) | 0.84 +/- 0.06 | 0.84 +/- 0.07 | 0.84 +/- 0.06 |


{'acc': 0.867463231086731, 'f1_macro': 0.8446657147200625, 'f1_weighted': 0.8687115181280782, 'acc_std': 0.0655361141104709, 'f1_macro_std': 0.09939143723563133, 'f1_weighted_std': 0.05932617979442903}

mrna only study.best_value=2.577811014967633, study.best_params={'lr': 0.00012898079159581874, 'l1_lambda': 0.00047366984296337946, 'l2_lambda': 0.0008509490658518463, 'proj_dim': 80, 'hidden_channels': 94}

In [7]:
mlp_eval(X, y, reg_type="inner_mat", n_trials=15)

Trial 0 / 15
Eval 1 / 5
Eval 2 / 5
Eval 3 / 5
Eval 4 / 5
Eval 5 / 5
[0.79591835 0.81632656 0.79591835 0.875      0.77083331]
[0.76975524 0.81948215 0.80430157 0.8947096  0.77475158]
[0.7956044  0.81951139 0.78980158 0.87082516 0.77207543]
{'acc': 0.8107993125915527, 'f1_macro': 0.8126000300289199, 'f1_weighted': 0.8095635923557335, 'acc_std': 0.035192175733458814, 'f1_macro_std': 0.04500308927522579, 'f1_weighted_std': 0.034183905959255544}
Trial 1 / 15
Eval 1 / 5
Eval 2 / 5


| MLP inner prod | 0.86 +/- 0.05 | 0.85 +/- 0.06 | 0.85 +/- 0.05 |  
| MLP (mrna only) | 0.87 +/- 0.06 | 0.87 +/- 0.07 | 0.87 +/- 0.06 |  
| MLP (mrna only) | 0.87 +/- 0.05 | 0.86 +/- 0.06 | 0.87 +/- 0.05 |  

{'acc': 0.8560374259948731, 'f1_macro': 0.8531890166223708, 'f1_weighted': 0.8540459975191783, 'acc_std': 0.05063744215757385, 'f1_macro_std': 0.06194529762727774, 'f1_weighted_std': 0.049910632703314174}

mrna only study.best_value=2.607328324144426, study.best_params={'lr': 8.077069025405151e-05, 'l1_lambda': 0.0004400516634407218, 'l2_lambda': 5.624746318975069e-05, 'proj_dim': 145, 'hidden_channels': 243}

# Results

| Method | ACC | F1 | F1 Weighted |
| --- | --- | --- | --- |
| KNN | 0.80 +/- 0.04 | 0.79 +/- 0.04 | 0.79 +/- 0.04 |
| LIN SVM w RFE | 0.84 +/- 0.01 | 0.84 +/- 0.02 | 0.84 +/- 0.01 |
| XGBoost | 0.88 +/- 0.02 | 0.88 +/- 0.03 | 0.88 +/- 0.02 |
| MLP no reg | 0.86 +/- 0.06 | 0.83 +/- 0.09 | 0.86 +/- 0.05 |
| MLP l1 reg | 0.87 +/- 0.06 | 0.85 +/- 0.09 | 0.87 +/- 0.05 |
| MLP inner prod | 0.86 +/- 0.05 | 0.85 +/- 0.06 | 0.85 +/- 0.05 |
| MOGONET (mrna, cnv, mirna) | 0.766 ± 0.025 | 0.742 ± 0.027| 0.768 ± 0.024|
| MOGONET (mrna, meth, mirna) | 0.824 ± 0.029 | 0.812 ± 0.039| 0.824 ± 0.032|
| MOGONET GAT (mrna only) | 0.86 +/- 0.04 | 0.84 +/- 0.05 | 0.86 +/- 0.04 |

In [133]:
# prepare 5 folds for cross validation
skf = StratifiedKFold(n_splits=5)

train_indices = []
val_indices = []
test_indices = []

for train_index, test_index in skf.split(X, y):
    train_indices.append(train_index)
    val_index, test_index = train_test_split(test_index, test_size=0.5, stratify=y[test_index])
    val_indices.append(val_index)
    test_indices.append(test_index)

best_val = torch.zeros(5, 3)
best_test = torch.zeros(5, 3)

In [150]:
mrna_mask = class_variational_selection(mrna_X[train_idx], y[train_idx], n_features=2000)

mrna_X_filt = torch.tensor(mrna_X[:, mrna_mask], dtype=torch.float32)

mrna_eps = 0.95
# meth_eps = 0.997
# cnv_eps = 0.9998
# mirna_eps = 0.962

# build adjacency matrix for each data type
A_mrna = cosine_similarity_matrix(mrna_X_filt)
A_mrna_bin = torch.where(A_mrna > mrna_eps, 1, 0) # - torch.eye(A_mrna.shape[0])
print(f"mrna mean degree {A_mrna_bin.sum(axis=1, dtype=torch.float32).mean()}")
print(
    f"Isolated samples = {(A_mrna_bin.sum(dim=1) == 1).sum()}"
)

mrna mean degree 32.97101593017578
Isolated samples = 131


In [108]:
from bipartite_gnn.graph_building import threshold_matrix

A_mrna = A_mrna.to(torch.float32)

threshold_matrix(A_mrna, self_connections=True, target_avg_degree=20)

0.75 tensor(482.8551)
0.875 tensor(370.6605)
0.9375 tensor(93.3520)
0.96875 tensor(2.1925)
0.953125 tensor(23.1408)
0.9609375 tensor(7.9689)
0.95703125 tensor(14.0062)
0.955078125 tensor(18.1180)
0.9541015625 tensor(20.3085)
Isolated samples = 0, avg degree = 20.308488845825195


tensor([[1., 1., 1.,  ..., 0., 0., 1.],
        [1., 1., 1.,  ..., 0., 0., 1.],
        [1., 1., 1.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 1., 0., 0.],
        [0., 0., 0.,  ..., 0., 1., 0.],
        [1., 1., 0.,  ..., 0., 0., 1.]])

In [161]:
for i in range(len(train_indices)):
    train_idx = train_indices[i]
    val_idx = val_indices[i]
    test_idx = test_indices[i]

    mrna_gene_names = mrna[:, 0].to_numpy()
    meth_gene_names = meth[:, 0].to_numpy()
    cna_gene_names = cnv[:, 0].to_numpy()
    mirna_gene_names = mirna[:, 0].to_numpy()

    mrna_mask = class_variational_selection(mrna_X[train_idx], y[train_idx], n_features=2000)
    meth_mask = class_variational_selection(meth_X[train_idx], y[train_idx], n_features=1000)
    cnv_mask = class_variational_selection(cnv_X[train_idx], y[train_idx], n_features=1000)
    mirna_mask = class_variational_selection(mirna_X[train_idx], y[train_idx], n_features=200)

    mrna_X_filt = mrna_X[:, mrna_mask]
    meth_X_filt = meth_X[:, meth_mask]
    cnv_X_filt = cnv_X[:, cnv_mask]
    mirna_X_filt = mirna_X[:, mirna_mask]

    # minmax scaling
    mms = MinMaxScaler()
    mms.fit(mrna_X_filt[train_idx])
    mrna_X_filt = torch.tensor(mms.transform(mrna_X_filt), dtype=torch.float32)

    mms = MinMaxScaler()
    mms.fit(meth_X_filt[train_idx])
    meth_X_filt = torch.tensor(mms.transform(meth_X_filt), dtype=torch.float32)

    mms = MinMaxScaler()
    mms.fit(cnv_X_filt[train_idx])
    cnv_X_filt = torch.tensor(mms.transform(cnv_X_filt), dtype=torch.float32)

    mms = MinMaxScaler()
    mms.fit(mirna_X_filt[train_idx])
    mirna_X_filt = torch.tensor(mms.transform(mirna_X_filt), dtype=torch.float32)


    mrna_gene_names = mrna_gene_names[mrna_mask]
    meth_gene_names = meth_gene_names[meth_mask]
    cna_gene_names = cna_gene_names[cnv_mask]
    mirna_gene_names = mirna_gene_names[mirna_mask]

    ##### BUILDING GRAPH #####

    # build graph for mogonet
    data = pyg.data.HeteroData()

    mrna_eps = 0.933
    meth_eps = 0.99
    cnv_eps = 0.99
    mirna_eps = 0.962

    # build adjacency matrix for each data type
    A_mrna = cosine_similarity_matrix(mrna_X_filt)
    A_mrna_bin = torch.where(A_mrna > mrna_eps, 1, 0) - torch.eye(A_mrna.shape[0])
    print(f"mrna mean degree {A_mrna_bin.sum(axis=1, dtype=torch.float32).mean()}")
    print(
        f"Isolated samples = {(A_mrna_bin.sum(dim=1) == 0).sum()}"
    )

    data['mrna'].x = mrna_X_filt
    data['mrna'].edge_index = dense_to_coo(A_mrna_bin)

    A_meth = cosine_similarity_matrix(meth_X_filt)
    A_meth_bin = torch.where(A_meth > meth_eps, 1, 0) - torch.eye(A_mrna.shape[0])
    A_meth_bin.sum(axis=1, dtype=torch.float32).mean()
    print(f"meth mean degree {A_meth_bin.sum(axis=1, dtype=torch.float32).mean()}")

    data['meth'].x = meth_X_filt
    data['meth'].edge_index = dense_to_coo(A_meth_bin)

    A_cnv = cosine_similarity_matrix(cnv_X_filt)
    A_cnv_bin = torch.where(A_cnv > cnv_eps, 1, 0) - torch.eye(A_mrna.shape[0])
    A_cnv_bin.sum(axis=1, dtype=torch.float32).mean() 
    print(f"cnv mean degree {A_cnv_bin.sum(axis=1, dtype=torch.float32).mean()}")

    data['cnv'].x = cnv_X_filt
    data['cnv'].edge_index = dense_to_coo(A_cnv_bin)

    A_mirna = cosine_similarity_matrix(mirna_X_filt)
    A_mirna_bin = torch.where(A_mirna > mirna_eps, 1, 0) - torch.eye(A_mrna.shape[0])
    A_mirna_bin.sum(axis=1, dtype=torch.float32).mean()
    print(f"mirna mean degree {A_mirna_bin.sum(axis=1, dtype=torch.float32).mean()}")

    data['mirna'].x = mirna_X_filt
    data['mirna'].edge_index = dense_to_coo(A_mirna_bin)

    data.y = torch.tensor(y)

    data.train_mask = torch.tensor(train_idx)
    data.val_mask = torch.tensor(val_idx)
    data.test_mask = torch.tensor(test_idx)

    ##### TRAINING MODEL #####

    model = MOGONET(
        omics=data.x_dict.keys(),
        in_channels=[data.x_dict[omics].shape[1] for omics in data.x_dict.keys()],
        hidden_channels=300,
        num_classes=4,
        encoder_type="gat",
        dropout=0.0,
    )

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=5e-4)

    ccounts = torch.unique(data.y[data.train_mask], return_counts=True)[1]
    inv_w = 1.0 / ccounts.float()
    class_weights = inv_w / inv_w.sum()

    trainer = GNNTrainer(
        model=model,
        loss_fn=torch.nn.CrossEntropyLoss(weight=class_weights),
        optimizer=optimizer,
        scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=15, min_lr=1e-5),
        params={
            "l1_lambda": None,
        }
    )

    bp = trainer.train(
        data=data,
        epochs=500,
        log_interval=50,
    )

    # print(bp)
    best_val[i] = bp[0]
    best_test[i] = bp[1]

mrna mean degree 10.786749839782715
Isolated samples = 257
meth mean degree 59.209110260009766
cnv mean degree 12.078675270080566
mirna mean degree 8.844720840454102
Epoch: 050:
Train Loss: 0.3424, Train Acc: 0.8394, Train F1 Macro: 0.8647, Train F1 Weighted: 0.8417
Val Loss: 0.5198, Val Acc: 0.7917, Val F1 Macro: 0.7911, Val F1 Weighted: 0.7990
Test Loss: 0.4351, Test Acc: 0.7959, Test F1 Macro: 0.8019, Test F1 Weighted: 0.8043
##################################################
Epoch: 100:
Train Loss: 0.0992, Train Acc: 0.9663, Train F1 Macro: 0.9729, Train F1 Weighted: 0.9663
Val Loss: 1.0371, Val Acc: 0.8750, Val F1 Macro: 0.8375, Val F1 Weighted: 0.8722
Test Loss: 0.3365, Test Acc: 0.8776, Test F1 Macro: 0.8685, Test F1 Weighted: 0.8800
##################################################
Epoch: 150:
Train Loss: 0.0783, Train Acc: 0.9715, Train F1 Macro: 0.9770, Train F1 Weighted: 0.9715
Val Loss: 1.2223, Val Acc: 0.8750, Val F1 Macro: 0.8375, Val F1 Weighted: 0.8722
Test Loss: 0.349

In [83]:
print(f"| MOGONET GCN (mrna only) | {best_test[:, 0].mean():.2f} +/- {best_test[:, 0].std():.2f} | {best_test[:, 1].mean():.2f} +/- {best_test[:, 1].std():.2f} | {best_test[:, 2].mean():.2f} +/- {best_test[:, 2].std():.2f} |")

| MOGONET GCN (mrna only) | 0.85 +/- 0.03 | 0.84 +/- 0.03 | 0.85 +/- 0.03 |


In [162]:
print(f"| MOGONET GAT (mrna only) | {best_val[:, 0].mean():.2f} +/- {best_val[:, 0].std():.2f} | {best_val[:, 1].mean():.2f} +/- {best_val[:, 1].std():.2f} | {best_val[:, 2].mean():.2f} +/- {best_val[:, 2].std():.2f} |")
print(f"| MOGONET GAT (mrna only) | {best_test[:, 0].mean():.2f} +/- {best_test[:, 0].std():.2f} | {best_test[:, 1].mean():.2f} +/- {best_test[:, 1].std():.2f} | {best_test[:, 2].mean():.2f} +/- {best_test[:, 2].std():.2f} |")

| MOGONET GAT (mrna only) | 0.87 +/- 0.05 | 0.87 +/- 0.05 | 0.87 +/- 0.05 |
| MOGONET GAT (mrna only) | 0.85 +/- 0.03 | 0.86 +/- 0.02 | 0.85 +/- 0.03 |


In [25]:
mrna

sample,TCGA-A1-A0SD-01,TCGA-A1-A0SE-01,TCGA-A1-A0SH-01,TCGA-A1-A0SJ-01,TCGA-A1-A0SK-01,TCGA-A1-A0SM-01,TCGA-A1-A0SO-01,TCGA-A2-A04N-01,TCGA-A2-A04P-01,TCGA-A2-A04Q-01,TCGA-A2-A04R-01,TCGA-A2-A04T-01,TCGA-A2-A04U-01,TCGA-A2-A04V-01,TCGA-A2-A04W-01,TCGA-A2-A04X-01,TCGA-A2-A04Y-01,TCGA-A2-A0CL-01,TCGA-A2-A0CM-01,TCGA-A2-A0CP-01,TCGA-A2-A0CQ-01,TCGA-A2-A0CS-01,TCGA-A2-A0CT-01,TCGA-A2-A0CU-01,TCGA-A2-A0CV-01,TCGA-A2-A0CW-01,TCGA-A2-A0D0-01,TCGA-A2-A0D1-01,TCGA-A2-A0D2-01,TCGA-A2-A0D3-01,TCGA-A2-A0D4-01,TCGA-A2-A0EM-01,TCGA-A2-A0EN-01,TCGA-A2-A0EO-01,TCGA-A2-A0EQ-01,TCGA-A2-A0ER-01,…,TCGA-E2-A14W-01,TCGA-E2-A14X-01,TCGA-E2-A14Y-01,TCGA-E2-A14Z-01,TCGA-E2-A150-01,TCGA-E2-A152-01,TCGA-E2-A153-01,TCGA-E2-A154-01,TCGA-E2-A155-01,TCGA-E2-A156-01,TCGA-E2-A158-01,TCGA-E2-A159-01,TCGA-E2-A15A-01,TCGA-E2-A15C-01,TCGA-E2-A15D-01,TCGA-E2-A15E-01,TCGA-E2-A15F-01,TCGA-E2-A15G-01,TCGA-E2-A15H-01,TCGA-E2-A15I-01,TCGA-E2-A15J-01,TCGA-E2-A15K-01,TCGA-E2-A15L-01,TCGA-E2-A15M-01,TCGA-E2-A15O-01,TCGA-E2-A15P-01,TCGA-E2-A15R-01,TCGA-E2-A15S-01,TCGA-E2-A15T-01,TCGA-E2-A1AZ-01,TCGA-E2-A1B0-01,TCGA-E2-A1B1-01,TCGA-E2-A1B4-01,TCGA-E2-A1B5-01,TCGA-E2-A1B6-01,TCGA-E2-A1BC-01,TCGA-E2-A1BD-01
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""ARHGEF10L""",9.1657,9.7179,8.8669,9.7663,9.8632,8.8234,8.6701,9.9831,10.5102,10.064,9.5425,10.449,9.4763,10.5333,8.9774,8.9665,9.3774,9.5693,11.4132,9.6413,10.3112,9.2703,9.0876,11.2247,8.9437,9.741,8.3209,11.1395,10.1611,9.2467,10.9353,9.3572,10.0545,9.778,10.2059,10.0794,…,8.9611,10.7636,9.9899,9.8046,11.2344,9.8236,9.984,9.2405,9.8415,10.3731,9.2199,10.1219,9.1705,10.0851,9.2262,9.7065,10.0334,10.0949,9.4848,9.1406,9.2531,9.9412,9.3713,8.453,9.376,9.1228,9.1934,10.0346,9.8209,9.1036,8.5746,9.3223,10.4367,8.9027,8.6965,9.0303,10.7991
"""HIF3A""",2.4935,3.3983,1.6631,6.1765,7.497,1.8177,8.9699,1.7558,6.9166,2.2799,1.4911,9.2573,6.3214,2.9105,2.5516,1.6612,3.6294,3.6849,0.0,4.8982,0.4407,2.3655,1.1726,2.9769,3.0291,3.35,1.7954,1.9355,2.1132,0.0,4.0158,1.4728,4.2849,2.5901,1.1658,1.8923,…,1.7638,4.8435,2.8783,2.9498,2.9782,2.3097,1.6892,1.9935,2.1329,0.7806,4.3358,4.1235,0.7135,0.848,3.8032,4.4089,0.8836,2.4616,1.1422,3.0117,0.4366,1.3397,1.6929,4.9786,0.8672,1.7922,0.3056,5.0409,0.3936,5.7418,1.3134,4.919,1.8654,4.8873,4.6748,6.9578,2.1772
"""RNF17""",0.0,0.0,0.4449,0.0,0.7051,1.178,0.0,0.0,0.6028,0.0,0.0,0.0,0.0,2.0906,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7.7299,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.8064,0.0,0.0,0.0,0.0,0.638,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.4333,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0372,0.0,0.0,1.4776,0.0,0.8114
"""RNF10""",11.7868,11.3219,11.8779,11.502,12.099,11.9934,11.6209,12.185,12.1741,11.7065,11.9362,11.3839,12.5415,12.22,12.0188,12.5742,12.0841,11.6492,12.0196,12.0527,12.2531,12.3055,11.1702,11.5801,12.0265,12.2589,11.8358,13.0469,11.7749,11.9473,12.1437,11.6602,12.0735,11.881,11.9862,11.965,…,12.3585,11.8525,11.7697,12.3889,11.3972,12.1153,11.7244,11.5464,11.9198,12.0953,11.8904,12.0105,12.2024,11.9198,11.9175,11.7502,12.0135,12.2843,12.3655,12.0196,11.9345,11.9704,11.88,11.7094,12.4611,12.4242,11.7385,12.4475,11.8324,11.416,11.7854,11.7005,12.428,12.6904,12.3403,11.8271,12.2783
"""RNF11""",11.0131,11.2617,11.8526,11.4405,9.9933,11.9332,11.0278,11.3242,11.2632,10.7574,10.4421,10.6897,10.9603,11.2698,11.9258,11.0061,10.9864,11.0977,11.1327,11.1117,11.1687,10.7255,11.4965,11.116,11.5781,10.6494,10.838,11.5712,10.9105,12.0452,10.1117,11.7259,10.9669,11.3562,11.3253,11.5178,…,11.3929,11.1539,11.0064,10.2678,11.758,11.8669,11.1408,10.3065,11.696,11.6226,10.6569,12.1943,10.6067,11.4468,11.066,11.5324,10.837,11.112,10.7207,11.3718,10.688,10.898,11.183,11.415,10.718,10.5873,10.1383,10.8017,11.1716,11.2437,10.9101,11

In [30]:
meth

gene,TCGA-A1-A0SD-01,TCGA-A1-A0SE-01,TCGA-A1-A0SH-01,TCGA-A1-A0SJ-01,TCGA-A1-A0SK-01,TCGA-A1-A0SM-01,TCGA-A1-A0SO-01,TCGA-A2-A04N-01,TCGA-A2-A04P-01,TCGA-A2-A04Q-01,TCGA-A2-A04R-01,TCGA-A2-A04T-01,TCGA-A2-A04U-01,TCGA-A2-A04V-01,TCGA-A2-A04W-01,TCGA-A2-A04X-01,TCGA-A2-A04Y-01,TCGA-A2-A0CL-01,TCGA-A2-A0CM-01,TCGA-A2-A0CP-01,TCGA-A2-A0CQ-01,TCGA-A2-A0CS-01,TCGA-A2-A0CT-01,TCGA-A2-A0CU-01,TCGA-A2-A0CV-01,TCGA-A2-A0CW-01,TCGA-A2-A0D0-01,TCGA-A2-A0D1-01,TCGA-A2-A0D2-01,TCGA-A2-A0D3-01,TCGA-A2-A0D4-01,TCGA-A2-A0EM-01,TCGA-A2-A0EN-01,TCGA-A2-A0EO-01,TCGA-A2-A0EQ-01,TCGA-A2-A0ER-01,…,TCGA-E2-A14W-01,TCGA-E2-A14X-01,TCGA-E2-A14Y-01,TCGA-E2-A14Z-01,TCGA-E2-A150-01,TCGA-E2-A152-01,TCGA-E2-A153-01,TCGA-E2-A154-01,TCGA-E2-A155-01,TCGA-E2-A156-01,TCGA-E2-A158-01,TCGA-E2-A159-01,TCGA-E2-A15A-01,TCGA-E2-A15C-01,TCGA-E2-A15D-01,TCGA-E2-A15E-01,TCGA-E2-A15F-01,TCGA-E2-A15G-01,TCGA-E2-A15H-01,TCGA-E2-A15I-01,TCGA-E2-A15J-01,TCGA-E2-A15K-01,TCGA-E2-A15L-01,TCGA-E2-A15M-01,TCGA-E2-A15O-01,TCGA-E2-A15P-01,TCGA-E2-A15R-01,TCGA-E2-A15S-01,TCGA-E2-A15T-01,TCGA-E2-A1AZ-01,TCGA-E2-A1B0-01,TCGA-E2-A1B1-01,TCGA-E2-A1B4-01,TCGA-E2-A1B5-01,TCGA-E2-A1B6-01,TCGA-E2-A1BC-01,TCGA-E2-A1BD-01
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""TMC2""",1.5235,8.087,7.3333,8.4696,8.2481,8.1269,8.7279,1.6294,1.6634,1.3048,5.6385,1.6858,1.8304,1.6379,1.7849,1.2717,1.5197,1.2425,1.5973,1.6295,0.8718,1.4092,9.5414,1.6181,1.5169,1.798,1.6772,1.8567,1.3112,1.276,0.704,1.3048,7.3193,1.1149,1.5163,1.2345,…,1.0135,1.6325,1.4252,1.241,1.172,1.6949,1.1047,1.71,0.9837,1.3829,0.625,1.5808,1.2221,1.255,1.5549,1.5098,1.6266,1.0942,1.4878,6.9093,8.0758,6.7213,0.9823,1.2805,1.4281,1.4518,1.8697,1.8668,1.4124,7.7729,8.8427,7.1233,6.202,7.9256,7.1273,7.0039,7.8961
"""CFD""",0.254,5.2768,4.7041,3.5122,1.0565,4.956,4.4651,0.0526,0.0726,0.1835,9.1483,0.0716,0.0676,0.0814,0.094,0.3496,0.264,0.1884,0.1214,0.0752,0.0664,0.1297,7.7855,0.0578,0.1519,0.0699,0.0675,0.0436,0.1128,0.0938,0.291,0.0326,5.2089,0.0543,0.1913,0.0572,…,0.0461,0.0815,0.0722,0.1521,0.0905,0.102,0.0941,0.0731,0.3922,0.0494,0.0477,0.0994,0.061,0.0575,0.0811,0.0684,0.2083,0.0584,0.08,3.745,6.8864,6.6225,0.0667,0.1414,0.133,0.1231,0.8016,0.6071,0.2317,3.4169,2.8814,3.9754,5.9056,4.758,3.6651,3.983,4.2721
"""E2F3""",0.0431,6.7408,6.7235,6.9288,6.1575,6.802,7.0211,0.0441,0.0312,0.043,6.6894,0.0323,0.0541,0.0423,0.0589,0.0417,0.0342,0.045,0.0422,0.0355,0.044,0.0715,6.9669,0.0416,0.0514,0.0612,0.0306,0.0375,0.0324,0.0507,0.0447,0.0415,6.6833,0.0433,0.0417,0.034,…,0.0363,0.041,0.0375,0.039,0.0585,0.0392,0.0563,0.0776,0.0909,0.0547,0.0532,0.0418,0.0388,0.0418,0.0573,0.0482,0.0731,0.0524,0.0492,6.6187,7.0347,6.7874,0.0589,0.0524,0.0402,0.0551,0.0493,0.0344,0.051,6.7312,6.6716,6.6688,7.0012,6.4084,6.7554,6.5313,6.5771
"""IARS""",0.0421,0.8795,0.9065,0.9167,0.8317,0.9234,0.9489,0.0596,0.0285,0.0431,0.9295,0.0326,0.0595,0.0476,0.0904,0.0453,0.0471,0.0512,0.0458,0.0447,0.0423,0.0616,0.941,0.0402,0.0425,0.0571,0.0509,0.0446,0.0362,0.0459,0.0401,0.0407,0.9149,0.0477,0.035,0.0381,…,0.069,0.0766,0.0541,0.0583,0.076,0.0516,0.0623,0.0605,0.0426,0.0626,0.0428,0.0685,0.0683,0.0558,0.0642,0.0547,0.0729,0.0507,0.0501,0.8997,0.916,0.9196,0.0492,0.0669,0.0633,0.0543,0.0506,0.06,0.0465,0.8639,0.8659,0.9102,0.9049,0.9332,0.9338,0.9256,0.8935
"""C15orf48""",0.0553,0.6827,0.5293,0.7318,0.2319,0.677,0.4034,0.0469,0.0433,0.0357,0.6212,0.0456,0.2557,0.4278,0.2031,0.0368,0.0389,0.0463,0.042,0.0336,0.0412,0.0548,0.5779,0.0516,0.0405,0.0539,0.0465,0.0319,0.0405,0.0557,0.0311,0.7483,0.5263,0.0511,0.0344,0.0421,…,0.0475,0.0559,0.0542,0.0479,0.0523,0.0473,0.0476,0.0459,0.0321,0.0527,0.0674,0.0668,0.0358,0.0689,0.0733,0.049,0.043,0.0469,0.0499,0.4768,0.7754,0.426,0.0494,0.0626,0.0428,0.0499,0.0434,0.0391,0.2833,0.64

In [33]:
cnv

Gene Symbol,TCGA-A1-A0SD-01,TCGA-A1-A0SE-01,TCGA-A1-A0SH-01,TCGA-A1-A0SJ-01,TCGA-A1-A0SK-01,TCGA-A1-A0SM-01,TCGA-A1-A0SO-01,TCGA-A2-A04N-01,TCGA-A2-A04P-01,TCGA-A2-A04Q-01,TCGA-A2-A04R-01,TCGA-A2-A04T-01,TCGA-A2-A04U-01,TCGA-A2-A04V-01,TCGA-A2-A04W-01,TCGA-A2-A04X-01,TCGA-A2-A04Y-01,TCGA-A2-A0CL-01,TCGA-A2-A0CM-01,TCGA-A2-A0CP-01,TCGA-A2-A0CQ-01,TCGA-A2-A0CS-01,TCGA-A2-A0CT-01,TCGA-A2-A0CU-01,TCGA-A2-A0CV-01,TCGA-A2-A0CW-01,TCGA-A2-A0D0-01,TCGA-A2-A0D1-01,TCGA-A2-A0D2-01,TCGA-A2-A0D3-01,TCGA-A2-A0D4-01,TCGA-A2-A0EM-01,TCGA-A2-A0EN-01,TCGA-A2-A0EO-01,TCGA-A2-A0EQ-01,TCGA-A2-A0ER-01,…,TCGA-E2-A14W-01,TCGA-E2-A14X-01,TCGA-E2-A14Y-01,TCGA-E2-A14Z-01,TCGA-E2-A150-01,TCGA-E2-A152-01,TCGA-E2-A153-01,TCGA-E2-A154-01,TCGA-E2-A155-01,TCGA-E2-A156-01,TCGA-E2-A158-01,TCGA-E2-A159-01,TCGA-E2-A15A-01,TCGA-E2-A15C-01,TCGA-E2-A15D-01,TCGA-E2-A15E-01,TCGA-E2-A15F-01,TCGA-E2-A15G-01,TCGA-E2-A15H-01,TCGA-E2-A15I-01,TCGA-E2-A15J-01,TCGA-E2-A15K-01,TCGA-E2-A15L-01,TCGA-E2-A15M-01,TCGA-E2-A15O-01,TCGA-E2-A15P-01,TCGA-E2-A15R-01,TCGA-E2-A15S-01,TCGA-E2-A15T-01,TCGA-E2-A1AZ-01,TCGA-E2-A1B0-01,TCGA-E2-A1B1-01,TCGA-E2-A1B4-01,TCGA-E2-A1B5-01,TCGA-E2-A1B6-01,TCGA-E2-A1BC-01,TCGA-E2-A1BD-01
str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,…,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
"""ACAP3""",-1,0,-1,0,1,-1,-1,0,1,0,-1,0,2,0,1,-1,1,0,1,0,0,-1,-1,0,0,-1,1,0,0,0,-1,0,-1,-1,0,0,…,0,1,0,1,1,0,0,-1,0,0,-1,1,-1,0,0,0,-1,0,-1,0,0,0,0,0,-1,0,-1,0,0,1,0,0,0,0,0,0,0
"""ACTRT2""",-1,0,-1,0,1,-1,-1,0,1,0,-1,0,2,0,1,-1,1,0,1,0,0,-1,-1,0,0,-1,1,0,0,0,-1,0,-1,-1,0,0,…,0,1,0,1,1,0,0,-1,0,0,-1,1,-1,0,0,0,-1,0,-1,0,0,0,0,0,-1,0,-1,0,0,1,0,0,0,0,0,0,0
"""AGRN""",-1,0,-1,0,1,-1,-1,0,1,0,-1,0,2,0,1,-1,1,0,1,0,0,-1,-1,0,0,-1,1,0,0,0,-1,0,-1,-1,0,0,…,0,1,0,1,1,0,0,-1,0,0,-1,1,-1,0,0,0,-1,0,-1,0,0,0,0,0,-1,0,-1,0,0,1,0,0,0,0,0,0,0
"""ANKRD65""",-1,0,-1,0,1,-1,-1,0,1,0,-1,0,2,0,1,-1,1,0,1,0,0,-1,-1,0,0,-1,1,0,0,0,-1,0,-1,-1,0,0,…,0,1,0,1,1,0,0,-1,0,0,-1,1,-1,0,0,0,-1,0,-1,0,0,0,0,0,-1,0,-1,0,0,1,0,0,0,0,0,0,0
"""ATAD3A""",-1,0,-1,0,1,-1,-1,0,1,0,-1,0,2,0,1,-1,1,0,1,0,0,-1,-1,0,0,-1,1,0,0,0,-1,0,-1,-1,0,0,…,0,1,0,1,1,0,0,-1,0,0,-1,1,-1,0,0,0,-1,0,-1,0,0,0,0,0,-1,0,-1,0,0,1,0,0,0,0,0,0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""IL9R|ENSG00000…",0,0,1,1,-1,0,1,0,0,0,-1,1,-1,0,0,0,0,0,1,0,0,-1,-1,0,0,-1,2,-1,1,0,0,0,0,-1,1,0,…,0,1,1,0,0,1,0,0,-1,0,1,0,0,0,1,0,0,0,-1,0,0,-1,0,0,0,-1,0,1,0,0,1,0,0,0,0,0,0
"""SPRY3|ENSG0000…",0,0,1,1,-1,0,1,0,0,0,-1,1,-1,0,0,0,0,0,1,0,0,-1,-1,0,0,-1,2,-1,1,0,0,0,0,-1,1,0,…,0,1,1,0,0,1,0,0,-1,0,1,0,0,0,1,0,0,0,-1,0,0,-1,0,0,0,-1,0,1,0,0,1,0,0,0,0,0,0
"""VAMP7|ENSG0000…",0,0,1,1,-1,0,1,0,0,0,-1,1,-1,0,0,0,0,0,1,0,0,-1,-1,0,0,-1,2,-1,1,0,0,0,0,-1,1,0,…,0,1,1,0,0,1,0,0,-1,0,1,0,0,0,1,0,0,0,-1,0,0,-1,0,0,0,-1,0,1,0,0,1,0,0,0,0,0,0


In [35]:
mirna

sample,TCGA-A1-A0SD-01,TCGA-A1-A0SE-01,TCGA-A1-A0SH-01,TCGA-A1-A0SJ-01,TCGA-A1-A0SK-01,TCGA-A1-A0SM-01,TCGA-A1-A0SO-01,TCGA-A2-A04N-01,TCGA-A2-A04P-01,TCGA-A2-A04Q-01,TCGA-A2-A04R-01,TCGA-A2-A04T-01,TCGA-A2-A04U-01,TCGA-A2-A04V-01,TCGA-A2-A04W-01,TCGA-A2-A04X-01,TCGA-A2-A04Y-01,TCGA-A2-A0CL-01,TCGA-A2-A0CM-01,TCGA-A2-A0CP-01,TCGA-A2-A0CQ-01,TCGA-A2-A0CS-01,TCGA-A2-A0CT-01,TCGA-A2-A0CU-01,TCGA-A2-A0CV-01,TCGA-A2-A0CW-01,TCGA-A2-A0D0-01,TCGA-A2-A0D1-01,TCGA-A2-A0D2-01,TCGA-A2-A0D3-01,TCGA-A2-A0D4-01,TCGA-A2-A0EM-01,TCGA-A2-A0EN-01,TCGA-A2-A0EO-01,TCGA-A2-A0EQ-01,TCGA-A2-A0ER-01,…,TCGA-E2-A14W-01,TCGA-E2-A14X-01,TCGA-E2-A14Y-01,TCGA-E2-A14Z-01,TCGA-E2-A150-01,TCGA-E2-A152-01,TCGA-E2-A153-01,TCGA-E2-A154-01,TCGA-E2-A155-01,TCGA-E2-A156-01,TCGA-E2-A158-01,TCGA-E2-A159-01,TCGA-E2-A15A-01,TCGA-E2-A15C-01,TCGA-E2-A15D-01,TCGA-E2-A15E-01,TCGA-E2-A15F-01,TCGA-E2-A15G-01,TCGA-E2-A15H-01,TCGA-E2-A15I-01,TCGA-E2-A15J-01,TCGA-E2-A15K-01,TCGA-E2-A15L-01,TCGA-E2-A15M-01,TCGA-E2-A15O-01,TCGA-E2-A15P-01,TCGA-E2-A15R-01,TCGA-E2-A15S-01,TCGA-E2-A15T-01,TCGA-E2-A1AZ-01,TCGA-E2-A1B0-01,TCGA-E2-A1B1-01,TCGA-E2-A1B4-01,TCGA-E2-A1B5-01,TCGA-E2-A1B6-01,TCGA-E2-A1BC-01,TCGA-E2-A1BD-01
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""MIMAT0000764""",5.071474,3.854013,2.382356,4.222913,4.509048,3.538197,5.514562,3.827273,5.163619,5.218112,4.663827,5.812459,5.45282,3.877629,4.736193,4.522835,4.994921,5.521553,5.213508,3.977732,6.225333,6.169947,4.411417,5.118287,5.321907,5.627718,4.19197,4.412894,5.710129,4.486352,4.627661,5.539772,4.089383,4.34764,6.039151,5.740298,…,6.79102,5.435611,5.455092,4.655462,5.723164,5.240847,5.637186,5.561047,6.447084,5.495307,5.274712,6.508886,5.347178,5.530293,6.172283,4.828744,5.078684,5.7478,5.645265,5.753738,6.449604,6.675172,3.577483,5.259366,4.19391,4.183046,4.498929,4.3758,4.747991,4.836478,5.601051,5.137768,5.300429,6.499371,6.952499,5.205349,5.808441
"""MIMAT0000761""",4.008542,3.422385,2.56466,4.381093,4.440702,4.811169,5.183405,1.528521,5.560937,5.844677,3.805917,7.132497,4.990257,5.062135,3.950807,4.565347,3.856579,4.487103,5.713741,3.762239,5.228245,3.890257,2.071117,5.634191,3.560319,4.29273,5.63753,5.485167,5.774008,4.330546,5.193082,5.401811,3.171776,3.12825,6.306729,5.242689,…,4.459391,4.867238,5.442092,4.001986,6.322038,4.767857,4.927768,4.389886,4.809694,4.881132,6.695727,6.448709,5.254399,4.836731,4.729531,4.440666,4.617312,4.309923,4.300865,3.466643,5.186683,5.48364,3.875092,4.278226,5.026157,3.31921,4.26946,4.714211,3.983686,5.313364,4.58701,3.246015,4.30574,4.598889,6.055791,3.898447,4.015216
"""MIMAT0000760""",5.210768,4.299733,4.463944,4.864602,5.026833,4.811129,5.371101,3.27361,4.203332,4.085836,4.724145,4.770054,5.859588,3.703502,5.810575,4.237739,5.311719,5.713766,3.276755,2.56518,3.947129,5.780914,4.298554,2.4394,4.566494,4.76149,5.380331,2.419569,2.707149,5.894983,4.500046,2.917995,4.645806,2.54591,3.771318,2.836636,…,5.218797,5.411936,4.313073,3.347708,5.282526,5.660289,5.531505,5.205205,6.123285,5.116064,6.320441,5.827211,5.568756,4.627838,5.388864,4.335751,4.488138,5.260209,5.254249,5.510268,5.575587,5.582197,5.0268,4.650923,5.6112,3.606065,5.231557,5.574656,5.664894,5.101961,4.644667,4.37875,4.069517,5.459581,5.56719,5.472708,4.866539
"""MIMAT0000763""",7.578522,7.59657,5.648522,8.309806,7.662071,10.914424,7.467316,6.032604,7.555345,8.832672,7.470783,7.461383,8.231619,8.793409,8.052233,6.888781,7.712954,9.568987,7.674214,7.11028,6.983853,8.212865,8.248575,7.781297,6.81931,7.816766,7.79798,5.965428,8.095695,7.728533,7.933645,10.091673,6.738124,6.266518,10.499198,11.578899,…,11.34812,7.068124,8.69743,7.208328,8.098249,8.439955,8.0972,6.254966,7.504197,10.291455,6.903545,6.96512,7.740712,6.882568,7.520524,9.318139,9.887819,7.111806,7.784409,6.030171

In [40]:
mirna_column = mirna['sample']
mirna_column = pl.DataFrame(mirna_column)

In [42]:
mirna_column.write_csv("brca_mirna_column.csv")

In [45]:
mirna_target_names = pl.read_csv("brca_mirna_target_names.csv", null_values=['NA'])

In [46]:
mirna_target_names

Accession,TargetName
str,str
"""MIMAT0000764""","""hsa-miR-339-5p…"
"""MIMAT0000761""","""hsa-miR-324-5p…"
"""MIMAT0000760""","""hsa-miR-331-3p…"
"""MIMAT0000763""","""hsa-miR-338-3p…"
"""MIMAT0000762""","""hsa-miR-324-3p…"
…,…
"""MIMAT0004673""","""hsa-miR-29c-5p…"
"""MIMAT0004672""","""hsa-miR-106b-3…"
"""MIMAT0004585""","""hsa-let-7i-3p"""


In [48]:
assert mirna_target_names['Accession'].to_list() == mirna['sample'].to_list()

In [50]:
mirna.with_columns(sample=mirna_target_names['TargetName'])

sample,TCGA-A1-A0SD-01,TCGA-A1-A0SE-01,TCGA-A1-A0SH-01,TCGA-A1-A0SJ-01,TCGA-A1-A0SK-01,TCGA-A1-A0SM-01,TCGA-A1-A0SO-01,TCGA-A2-A04N-01,TCGA-A2-A04P-01,TCGA-A2-A04Q-01,TCGA-A2-A04R-01,TCGA-A2-A04T-01,TCGA-A2-A04U-01,TCGA-A2-A04V-01,TCGA-A2-A04W-01,TCGA-A2-A04X-01,TCGA-A2-A04Y-01,TCGA-A2-A0CL-01,TCGA-A2-A0CM-01,TCGA-A2-A0CP-01,TCGA-A2-A0CQ-01,TCGA-A2-A0CS-01,TCGA-A2-A0CT-01,TCGA-A2-A0CU-01,TCGA-A2-A0CV-01,TCGA-A2-A0CW-01,TCGA-A2-A0D0-01,TCGA-A2-A0D1-01,TCGA-A2-A0D2-01,TCGA-A2-A0D3-01,TCGA-A2-A0D4-01,TCGA-A2-A0EM-01,TCGA-A2-A0EN-01,TCGA-A2-A0EO-01,TCGA-A2-A0EQ-01,TCGA-A2-A0ER-01,…,TCGA-E2-A14W-01,TCGA-E2-A14X-01,TCGA-E2-A14Y-01,TCGA-E2-A14Z-01,TCGA-E2-A150-01,TCGA-E2-A152-01,TCGA-E2-A153-01,TCGA-E2-A154-01,TCGA-E2-A155-01,TCGA-E2-A156-01,TCGA-E2-A158-01,TCGA-E2-A159-01,TCGA-E2-A15A-01,TCGA-E2-A15C-01,TCGA-E2-A15D-01,TCGA-E2-A15E-01,TCGA-E2-A15F-01,TCGA-E2-A15G-01,TCGA-E2-A15H-01,TCGA-E2-A15I-01,TCGA-E2-A15J-01,TCGA-E2-A15K-01,TCGA-E2-A15L-01,TCGA-E2-A15M-01,TCGA-E2-A15O-01,TCGA-E2-A15P-01,TCGA-E2-A15R-01,TCGA-E2-A15S-01,TCGA-E2-A15T-01,TCGA-E2-A1AZ-01,TCGA-E2-A1B0-01,TCGA-E2-A1B1-01,TCGA-E2-A1B4-01,TCGA-E2-A1B5-01,TCGA-E2-A1B6-01,TCGA-E2-A1BC-01,TCGA-E2-A1BD-01
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""hsa-miR-339-5p…",5.071474,3.854013,2.382356,4.222913,4.509048,3.538197,5.514562,3.827273,5.163619,5.218112,4.663827,5.812459,5.45282,3.877629,4.736193,4.522835,4.994921,5.521553,5.213508,3.977732,6.225333,6.169947,4.411417,5.118287,5.321907,5.627718,4.19197,4.412894,5.710129,4.486352,4.627661,5.539772,4.089383,4.34764,6.039151,5.740298,…,6.79102,5.435611,5.455092,4.655462,5.723164,5.240847,5.637186,5.561047,6.447084,5.495307,5.274712,6.508886,5.347178,5.530293,6.172283,4.828744,5.078684,5.7478,5.645265,5.753738,6.449604,6.675172,3.577483,5.259366,4.19391,4.183046,4.498929,4.3758,4.747991,4.836478,5.601051,5.137768,5.300429,6.499371,6.952499,5.205349,5.808441
"""hsa-miR-324-5p…",4.008542,3.422385,2.56466,4.381093,4.440702,4.811169,5.183405,1.528521,5.560937,5.844677,3.805917,7.132497,4.990257,5.062135,3.950807,4.565347,3.856579,4.487103,5.713741,3.762239,5.228245,3.890257,2.071117,5.634191,3.560319,4.29273,5.63753,5.485167,5.774008,4.330546,5.193082,5.401811,3.171776,3.12825,6.306729,5.242689,…,4.459391,4.867238,5.442092,4.001986,6.322038,4.767857,4.927768,4.389886,4.809694,4.881132,6.695727,6.448709,5.254399,4.836731,4.729531,4.440666,4.617312,4.309923,4.300865,3.466643,5.186683,5.48364,3.875092,4.278226,5.026157,3.31921,4.26946,4.714211,3.983686,5.313364,4.58701,3.246015,4.30574,4.598889,6.055791,3.898447,4.015216
"""hsa-miR-331-3p…",5.210768,4.299733,4.463944,4.864602,5.026833,4.811129,5.371101,3.27361,4.203332,4.085836,4.724145,4.770054,5.859588,3.703502,5.810575,4.237739,5.311719,5.713766,3.276755,2.56518,3.947129,5.780914,4.298554,2.4394,4.566494,4.76149,5.380331,2.419569,2.707149,5.894983,4.500046,2.917995,4.645806,2.54591,3.771318,2.836636,…,5.218797,5.411936,4.313073,3.347708,5.282526,5.660289,5.531505,5.205205,6.123285,5.116064,6.320441,5.827211,5.568756,4.627838,5.388864,4.335751,4.488138,5.260209,5.254249,5.510268,5.575587,5.582197,5.0268,4.650923,5.6112,3.606065,5.231557,5.574656,5.664894,5.101961,4.644667,4.37875,4.069517,5.459581,5.56719,5.472708,4.866539
"""hsa-miR-338-3p…",7.578522,7.59657,5.648522,8.309806,7.662071,10.914424,7.467316,6.032604,7.555345,8.832672,7.470783,7.461383,8.231619,8.793409,8.052233,6.888781,7.712954,9.568987,7.674214,7.11028,6.983853,8.212865,8.248575,7.781297,6.81931,7.816766,7.79798,5.965428,8.095695,7.728533,7.933645,10.091673,6.738124,6.266518,10.499198,11.578899,…,11.34812,7.068124,8.69743,7.208328,8.098249,8.439955,8.0972,6.254966,7.504197,10.291455,6.903545,6.96512,7.740712,6.882568,7.520524,9.318139,9.887819,7.111806,7.784409,6.03

In [10]:
mirna

sample,TCGA-A1-A0SD-01,TCGA-A1-A0SE-01,TCGA-A1-A0SH-01,TCGA-A1-A0SJ-01,TCGA-A1-A0SK-01,TCGA-A1-A0SM-01,TCGA-A1-A0SO-01,TCGA-A2-A04N-01,TCGA-A2-A04P-01,TCGA-A2-A04Q-01,TCGA-A2-A04R-01,TCGA-A2-A04T-01,TCGA-A2-A04U-01,TCGA-A2-A04V-01,TCGA-A2-A04W-01,TCGA-A2-A04X-01,TCGA-A2-A04Y-01,TCGA-A2-A0CL-01,TCGA-A2-A0CM-01,TCGA-A2-A0CP-01,TCGA-A2-A0CQ-01,TCGA-A2-A0CS-01,TCGA-A2-A0CT-01,TCGA-A2-A0CU-01,TCGA-A2-A0CV-01,TCGA-A2-A0CW-01,TCGA-A2-A0D0-01,TCGA-A2-A0D1-01,TCGA-A2-A0D2-01,TCGA-A2-A0D3-01,TCGA-A2-A0D4-01,TCGA-A2-A0EM-01,TCGA-A2-A0EN-01,TCGA-A2-A0EO-01,TCGA-A2-A0EQ-01,TCGA-A2-A0ER-01,…,TCGA-E2-A14W-01,TCGA-E2-A14X-01,TCGA-E2-A14Y-01,TCGA-E2-A14Z-01,TCGA-E2-A150-01,TCGA-E2-A152-01,TCGA-E2-A153-01,TCGA-E2-A154-01,TCGA-E2-A155-01,TCGA-E2-A156-01,TCGA-E2-A158-01,TCGA-E2-A159-01,TCGA-E2-A15A-01,TCGA-E2-A15C-01,TCGA-E2-A15D-01,TCGA-E2-A15E-01,TCGA-E2-A15F-01,TCGA-E2-A15G-01,TCGA-E2-A15H-01,TCGA-E2-A15I-01,TCGA-E2-A15J-01,TCGA-E2-A15K-01,TCGA-E2-A15L-01,TCGA-E2-A15M-01,TCGA-E2-A15O-01,TCGA-E2-A15P-01,TCGA-E2-A15R-01,TCGA-E2-A15S-01,TCGA-E2-A15T-01,TCGA-E2-A1AZ-01,TCGA-E2-A1B0-01,TCGA-E2-A1B1-01,TCGA-E2-A1B4-01,TCGA-E2-A1B5-01,TCGA-E2-A1B6-01,TCGA-E2-A1BC-01,TCGA-E2-A1BD-01
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""MIMAT0000764""",5.071474,3.854013,2.382356,4.222913,4.509048,3.538197,5.514562,3.827273,5.163619,5.218112,4.663827,5.812459,5.45282,3.877629,4.736193,4.522835,4.994921,5.521553,5.213508,3.977732,6.225333,6.169947,4.411417,5.118287,5.321907,5.627718,4.19197,4.412894,5.710129,4.486352,4.627661,5.539772,4.089383,4.34764,6.039151,5.740298,…,6.79102,5.435611,5.455092,4.655462,5.723164,5.240847,5.637186,5.561047,6.447084,5.495307,5.274712,6.508886,5.347178,5.530293,6.172283,4.828744,5.078684,5.7478,5.645265,5.753738,6.449604,6.675172,3.577483,5.259366,4.19391,4.183046,4.498929,4.3758,4.747991,4.836478,5.601051,5.137768,5.300429,6.499371,6.952499,5.205349,5.808441
"""MIMAT0000761""",4.008542,3.422385,2.56466,4.381093,4.440702,4.811169,5.183405,1.528521,5.560937,5.844677,3.805917,7.132497,4.990257,5.062135,3.950807,4.565347,3.856579,4.487103,5.713741,3.762239,5.228245,3.890257,2.071117,5.634191,3.560319,4.29273,5.63753,5.485167,5.774008,4.330546,5.193082,5.401811,3.171776,3.12825,6.306729,5.242689,…,4.459391,4.867238,5.442092,4.001986,6.322038,4.767857,4.927768,4.389886,4.809694,4.881132,6.695727,6.448709,5.254399,4.836731,4.729531,4.440666,4.617312,4.309923,4.300865,3.466643,5.186683,5.48364,3.875092,4.278226,5.026157,3.31921,4.26946,4.714211,3.983686,5.313364,4.58701,3.246015,4.30574,4.598889,6.055791,3.898447,4.015216
"""MIMAT0000760""",5.210768,4.299733,4.463944,4.864602,5.026833,4.811129,5.371101,3.27361,4.203332,4.085836,4.724145,4.770054,5.859588,3.703502,5.810575,4.237739,5.311719,5.713766,3.276755,2.56518,3.947129,5.780914,4.298554,2.4394,4.566494,4.76149,5.380331,2.419569,2.707149,5.894983,4.500046,2.917995,4.645806,2.54591,3.771318,2.836636,…,5.218797,5.411936,4.313073,3.347708,5.282526,5.660289,5.531505,5.205205,6.123285,5.116064,6.320441,5.827211,5.568756,4.627838,5.388864,4.335751,4.488138,5.260209,5.254249,5.510268,5.575587,5.582197,5.0268,4.650923,5.6112,3.606065,5.231557,5.574656,5.664894,5.101961,4.644667,4.37875,4.069517,5.459581,5.56719,5.472708,4.866539
"""MIMAT0000763""",7.578522,7.59657,5.648522,8.309806,7.662071,10.914424,7.467316,6.032604,7.555345,8.832672,7.470783,7.461383,8.231619,8.793409,8.052233,6.888781,7.712954,9.568987,7.674214,7.11028,6.983853,8.212865,8.248575,7.781297,6.81931,7.816766,7.79798,5.965428,8.095695,7.728533,7.933645,10.091673,6.738124,6.266518,10.499198,11.578899,…,11.34812,7.068124,8.69743,7.208328,8.098249,8.439955,8.0972,6.254966,7.504197,10.291455,6.903545,6.96512,7.740712,6.882568,7.520524,9.318139,9.887819,7.111806,7.784409,6.030171

In [12]:
mirna_samples = mirna['sample'].to_list()

In [16]:
mirbase_conver = pl.read_csv("miRBaseConverter_Result_all.csv", null_values=['NA'])

In [18]:
assert mirna_samples == mirbase_conver['Accession'].to_list()

In [20]:
mirna = mirna.with_columns(sample=mirbase_conver['TargetName'])

In [21]:
mirna

sample,TCGA-A1-A0SD-01,TCGA-A1-A0SE-01,TCGA-A1-A0SH-01,TCGA-A1-A0SJ-01,TCGA-A1-A0SK-01,TCGA-A1-A0SM-01,TCGA-A1-A0SO-01,TCGA-A2-A04N-01,TCGA-A2-A04P-01,TCGA-A2-A04Q-01,TCGA-A2-A04R-01,TCGA-A2-A04T-01,TCGA-A2-A04U-01,TCGA-A2-A04V-01,TCGA-A2-A04W-01,TCGA-A2-A04X-01,TCGA-A2-A04Y-01,TCGA-A2-A0CL-01,TCGA-A2-A0CM-01,TCGA-A2-A0CP-01,TCGA-A2-A0CQ-01,TCGA-A2-A0CS-01,TCGA-A2-A0CT-01,TCGA-A2-A0CU-01,TCGA-A2-A0CV-01,TCGA-A2-A0CW-01,TCGA-A2-A0D0-01,TCGA-A2-A0D1-01,TCGA-A2-A0D2-01,TCGA-A2-A0D3-01,TCGA-A2-A0D4-01,TCGA-A2-A0EM-01,TCGA-A2-A0EN-01,TCGA-A2-A0EO-01,TCGA-A2-A0EQ-01,TCGA-A2-A0ER-01,…,TCGA-E2-A14W-01,TCGA-E2-A14X-01,TCGA-E2-A14Y-01,TCGA-E2-A14Z-01,TCGA-E2-A150-01,TCGA-E2-A152-01,TCGA-E2-A153-01,TCGA-E2-A154-01,TCGA-E2-A155-01,TCGA-E2-A156-01,TCGA-E2-A158-01,TCGA-E2-A159-01,TCGA-E2-A15A-01,TCGA-E2-A15C-01,TCGA-E2-A15D-01,TCGA-E2-A15E-01,TCGA-E2-A15F-01,TCGA-E2-A15G-01,TCGA-E2-A15H-01,TCGA-E2-A15I-01,TCGA-E2-A15J-01,TCGA-E2-A15K-01,TCGA-E2-A15L-01,TCGA-E2-A15M-01,TCGA-E2-A15O-01,TCGA-E2-A15P-01,TCGA-E2-A15R-01,TCGA-E2-A15S-01,TCGA-E2-A15T-01,TCGA-E2-A1AZ-01,TCGA-E2-A1B0-01,TCGA-E2-A1B1-01,TCGA-E2-A1B4-01,TCGA-E2-A1B5-01,TCGA-E2-A1B6-01,TCGA-E2-A1BC-01,TCGA-E2-A1BD-01
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""hsa-miR-339-5p…",5.071474,3.854013,2.382356,4.222913,4.509048,3.538197,5.514562,3.827273,5.163619,5.218112,4.663827,5.812459,5.45282,3.877629,4.736193,4.522835,4.994921,5.521553,5.213508,3.977732,6.225333,6.169947,4.411417,5.118287,5.321907,5.627718,4.19197,4.412894,5.710129,4.486352,4.627661,5.539772,4.089383,4.34764,6.039151,5.740298,…,6.79102,5.435611,5.455092,4.655462,5.723164,5.240847,5.637186,5.561047,6.447084,5.495307,5.274712,6.508886,5.347178,5.530293,6.172283,4.828744,5.078684,5.7478,5.645265,5.753738,6.449604,6.675172,3.577483,5.259366,4.19391,4.183046,4.498929,4.3758,4.747991,4.836478,5.601051,5.137768,5.300429,6.499371,6.952499,5.205349,5.808441
"""hsa-miR-324-5p…",4.008542,3.422385,2.56466,4.381093,4.440702,4.811169,5.183405,1.528521,5.560937,5.844677,3.805917,7.132497,4.990257,5.062135,3.950807,4.565347,3.856579,4.487103,5.713741,3.762239,5.228245,3.890257,2.071117,5.634191,3.560319,4.29273,5.63753,5.485167,5.774008,4.330546,5.193082,5.401811,3.171776,3.12825,6.306729,5.242689,…,4.459391,4.867238,5.442092,4.001986,6.322038,4.767857,4.927768,4.389886,4.809694,4.881132,6.695727,6.448709,5.254399,4.836731,4.729531,4.440666,4.617312,4.309923,4.300865,3.466643,5.186683,5.48364,3.875092,4.278226,5.026157,3.31921,4.26946,4.714211,3.983686,5.313364,4.58701,3.246015,4.30574,4.598889,6.055791,3.898447,4.015216
"""hsa-miR-331-3p…",5.210768,4.299733,4.463944,4.864602,5.026833,4.811129,5.371101,3.27361,4.203332,4.085836,4.724145,4.770054,5.859588,3.703502,5.810575,4.237739,5.311719,5.713766,3.276755,2.56518,3.947129,5.780914,4.298554,2.4394,4.566494,4.76149,5.380331,2.419569,2.707149,5.894983,4.500046,2.917995,4.645806,2.54591,3.771318,2.836636,…,5.218797,5.411936,4.313073,3.347708,5.282526,5.660289,5.531505,5.205205,6.123285,5.116064,6.320441,5.827211,5.568756,4.627838,5.388864,4.335751,4.488138,5.260209,5.254249,5.510268,5.575587,5.582197,5.0268,4.650923,5.6112,3.606065,5.231557,5.574656,5.664894,5.101961,4.644667,4.37875,4.069517,5.459581,5.56719,5.472708,4.866539
"""hsa-miR-338-3p…",7.578522,7.59657,5.648522,8.309806,7.662071,10.914424,7.467316,6.032604,7.555345,8.832672,7.470783,7.461383,8.231619,8.793409,8.052233,6.888781,7.712954,9.568987,7.674214,7.11028,6.983853,8.212865,8.248575,7.781297,6.81931,7.816766,7.79798,5.965428,8.095695,7.728533,7.933645,10.091673,6.738124,6.266518,10.499198,11.578899,…,11.34812,7.068124,8.69743,7.208328,8.098249,8.439955,8.0972,6.254966,7.504197,10.291455,6.903545,6.96512,7.740712,6.882568,7.520524,9.318139,9.887819,7.111806,7.784409,6.03

In [17]:
mirbase_conver

Accession,TargetName
str,str
"""MIMAT0000764""","""hsa-miR-339-5p…"
"""MIMAT0000761""","""hsa-miR-324-5p…"
"""MIMAT0000760""","""hsa-miR-331-3p…"
"""MIMAT0000763""","""hsa-miR-338-3p…"
"""MIMAT0000762""","""hsa-miR-324-3p…"
…,…
"""MIMAT0004673""","""hsa-miR-29c-5p…"
"""MIMAT0004672""","""hsa-miR-106b-3…"
"""MIMAT0004585""","""hsa-let-7i-3p"""


In [30]:
mirna = mirna.drop_nulls()

In [31]:
mirna

sample,TCGA-A1-A0SD-01,TCGA-A1-A0SE-01,TCGA-A1-A0SH-01,TCGA-A1-A0SJ-01,TCGA-A1-A0SK-01,TCGA-A1-A0SM-01,TCGA-A1-A0SO-01,TCGA-A2-A04N-01,TCGA-A2-A04P-01,TCGA-A2-A04Q-01,TCGA-A2-A04R-01,TCGA-A2-A04T-01,TCGA-A2-A04U-01,TCGA-A2-A04V-01,TCGA-A2-A04W-01,TCGA-A2-A04X-01,TCGA-A2-A04Y-01,TCGA-A2-A0CL-01,TCGA-A2-A0CM-01,TCGA-A2-A0CP-01,TCGA-A2-A0CQ-01,TCGA-A2-A0CS-01,TCGA-A2-A0CT-01,TCGA-A2-A0CU-01,TCGA-A2-A0CV-01,TCGA-A2-A0CW-01,TCGA-A2-A0D0-01,TCGA-A2-A0D1-01,TCGA-A2-A0D2-01,TCGA-A2-A0D3-01,TCGA-A2-A0D4-01,TCGA-A2-A0EM-01,TCGA-A2-A0EN-01,TCGA-A2-A0EO-01,TCGA-A2-A0EQ-01,TCGA-A2-A0ER-01,…,TCGA-E2-A14W-01,TCGA-E2-A14X-01,TCGA-E2-A14Y-01,TCGA-E2-A14Z-01,TCGA-E2-A150-01,TCGA-E2-A152-01,TCGA-E2-A153-01,TCGA-E2-A154-01,TCGA-E2-A155-01,TCGA-E2-A156-01,TCGA-E2-A158-01,TCGA-E2-A159-01,TCGA-E2-A15A-01,TCGA-E2-A15C-01,TCGA-E2-A15D-01,TCGA-E2-A15E-01,TCGA-E2-A15F-01,TCGA-E2-A15G-01,TCGA-E2-A15H-01,TCGA-E2-A15I-01,TCGA-E2-A15J-01,TCGA-E2-A15K-01,TCGA-E2-A15L-01,TCGA-E2-A15M-01,TCGA-E2-A15O-01,TCGA-E2-A15P-01,TCGA-E2-A15R-01,TCGA-E2-A15S-01,TCGA-E2-A15T-01,TCGA-E2-A1AZ-01,TCGA-E2-A1B0-01,TCGA-E2-A1B1-01,TCGA-E2-A1B4-01,TCGA-E2-A1B5-01,TCGA-E2-A1B6-01,TCGA-E2-A1BC-01,TCGA-E2-A1BD-01
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""hsa-miR-339-5p…",5.071474,3.854013,2.382356,4.222913,4.509048,3.538197,5.514562,3.827273,5.163619,5.218112,4.663827,5.812459,5.45282,3.877629,4.736193,4.522835,4.994921,5.521553,5.213508,3.977732,6.225333,6.169947,4.411417,5.118287,5.321907,5.627718,4.19197,4.412894,5.710129,4.486352,4.627661,5.539772,4.089383,4.34764,6.039151,5.740298,…,6.79102,5.435611,5.455092,4.655462,5.723164,5.240847,5.637186,5.561047,6.447084,5.495307,5.274712,6.508886,5.347178,5.530293,6.172283,4.828744,5.078684,5.7478,5.645265,5.753738,6.449604,6.675172,3.577483,5.259366,4.19391,4.183046,4.498929,4.3758,4.747991,4.836478,5.601051,5.137768,5.300429,6.499371,6.952499,5.205349,5.808441
"""hsa-miR-324-5p…",4.008542,3.422385,2.56466,4.381093,4.440702,4.811169,5.183405,1.528521,5.560937,5.844677,3.805917,7.132497,4.990257,5.062135,3.950807,4.565347,3.856579,4.487103,5.713741,3.762239,5.228245,3.890257,2.071117,5.634191,3.560319,4.29273,5.63753,5.485167,5.774008,4.330546,5.193082,5.401811,3.171776,3.12825,6.306729,5.242689,…,4.459391,4.867238,5.442092,4.001986,6.322038,4.767857,4.927768,4.389886,4.809694,4.881132,6.695727,6.448709,5.254399,4.836731,4.729531,4.440666,4.617312,4.309923,4.300865,3.466643,5.186683,5.48364,3.875092,4.278226,5.026157,3.31921,4.26946,4.714211,3.983686,5.313364,4.58701,3.246015,4.30574,4.598889,6.055791,3.898447,4.015216
"""hsa-miR-331-3p…",5.210768,4.299733,4.463944,4.864602,5.026833,4.811129,5.371101,3.27361,4.203332,4.085836,4.724145,4.770054,5.859588,3.703502,5.810575,4.237739,5.311719,5.713766,3.276755,2.56518,3.947129,5.780914,4.298554,2.4394,4.566494,4.76149,5.380331,2.419569,2.707149,5.894983,4.500046,2.917995,4.645806,2.54591,3.771318,2.836636,…,5.218797,5.411936,4.313073,3.347708,5.282526,5.660289,5.531505,5.205205,6.123285,5.116064,6.320441,5.827211,5.568756,4.627838,5.388864,4.335751,4.488138,5.260209,5.254249,5.510268,5.575587,5.582197,5.0268,4.650923,5.6112,3.606065,5.231557,5.574656,5.664894,5.101961,4.644667,4.37875,4.069517,5.459581,5.56719,5.472708,4.866539
"""hsa-miR-338-3p…",7.578522,7.59657,5.648522,8.309806,7.662071,10.914424,7.467316,6.032604,7.555345,8.832672,7.470783,7.461383,8.231619,8.793409,8.052233,6.888781,7.712954,9.568987,7.674214,7.11028,6.983853,8.212865,8.248575,7.781297,6.81931,7.816766,7.79798,5.965428,8.095695,7.728533,7.933645,10.091673,6.738124,6.266518,10.499198,11.578899,…,11.34812,7.068124,8.69743,7.208328,8.098249,8.439955,8.0972,6.254966,7.504197,10.291455,6.903545,6.96512,7.740712,6.882568,7.520524,9.318139,9.887819,7.111806,7.784409,6.03

In [32]:
skf = StratifiedKFold(n_splits=5)

for i, (train_idx, test_idx) in enumerate(skf.split(np.zeros(len(y)), y)):
    # print(train_idx.shape, test_idx.shape)
    # print(train_idx, test_idx)
    # print(np.unique(y[train_idx], return_counts=True))
    # print(np.unique(y[test_idx], return_counts=True))

    mirna_selected = mrmr_selection(
        X_df=mirna,
        y=y,
        train_mask=train_idx,
        n_features=200,
        n_preselected_features=None,
        feature_col_name="sample",
    )

    mirna_selected.write_csv(f"BRCA_PROCESSED_DATA/mrmr_selected/mirna/mirna_fold_{i}.csv")

(386, 229)


100%|██████████| 200/200 [00:03<00:00, 64.93it/s] 


(386, 229)


100%|██████████| 200/200 [00:03<00:00, 64.93it/s] 


(386, 229)


100%|██████████| 200/200 [00:03<00:00, 63.93it/s] 


(387, 229)


100%|██████████| 200/200 [00:03<00:00, 64.53it/s] 


(387, 229)


100%|██████████| 200/200 [00:03<00:00, 64.47it/s] 


In [8]:
val, test = mogonet_eval(
    input_omics={
        "mrna": mrna,
        "meth": meth,
        "cnv": cnv,
        "mirna": mirna,
    },
    # n_input_features={
        # "mrna": 3000,
        # "meth": 1000,
        # "cnv": 1000,
        # "mirna": 200,
    # },
    mrmr_selection_params={
        'mrmr_path' : "BRCA_PROCESSED_DATA/mrmr_selected",
    },
    y=y,
    params={
        "encoder_hidden_channels": {
            "mrna": 70,
            "meth": 70,
            "cnv": 70,
            "mirna": 70,
        },
        "encoder_type": "gat",
        "num_classes": 4,
        "graph_style": "threshold",
        "avg_degree": 15,
        # "graph_style": "knn",
        # "knn": 7,
        "dropout": 0.05,
        "epochs": 400,
        "log_interval": 50,
        "save_best_model": False,
        "integrator_type" : "vcdn",
        "integration_dim" : 4,
        "self_connections": True,
    }
)

dict_keys(['mrna', 'meth', 'cnv', 'mirna'])
Fold 1 / 5
dict_keys(['mrna', 'meth', 'cnv', 'mirna'])
Building graph for mrna
0.75 tensor(413.0704)
0.875 tensor(312.4948)
0.9375 tensor(177.3064)
0.96875 tensor(22.7640)
0.984375 tensor(1.1159)
0.9765625 tensor(4.9006)
0.97265625 tensor(11.8323)
0.970703125 tensor(16.7681)
0.9716796875 tensor(14.2298)
0.97119140625 tensor(15.4596)
Isolated samples = 212, avg degree = 15.459627151489258
Building graph for meth
0.75 tensor(261.2278)
0.875 tensor(161.4555)
0.9375 tensor(81.2319)
0.96875 tensor(46.4451)
0.984375 tensor(13.0083)
0.9765625 tensor(31.0207)
0.98046875 tensor(22.5114)
0.982421875 tensor(17.7950)
0.9833984375 tensor(15.2360)
Isolated samples = 322, avg degree = 15.236024856567383
Building graph for cnv
0.75 tensor(343.2360)
0.875 tensor(186.4120)
0.9375 tensor(74.6729)
0.96875 tensor(27.6294)
0.984375 tensor(11.2733)
0.9765625 tensor(18.2091)
0.98046875 tensor(14.7640)
Isolated samples = 213, avg degree = 14.763975143432617
Building 

- increasing the integration dimension is trash
- vcdn is better, than the attention integrator
- thresholding graph is better than the knn graph even though it can be fairly sparse

In [9]:
print("MOGONET GAT (mrna, mirna, meth) results:")
print(f"| MOGONET GAT val | {val[:, 0].mean():.2f} +/- {val[:, 0].std():.2f} | {val[:, 1].mean():.2f} +/- {val[:, 1].std():.2f} | {val[:, 2].mean():.2f} +/- {val[:, 2].std():.2f} |")
print(f"| MOGONET GAT test | {test[:, 0].mean():.2f} +/- {test[:, 0].std():.2f} | {test[:, 1].mean():.2f} +/- {test[:, 1].std():.2f} | {test[:, 2].mean():.2f} +/- {test[:, 2].std():.2f} |")

MOGONET GAT (mrna, mirna, meth) results:
| MOGONET GAT val | 0.88 +/- 0.07 | 0.88 +/- 0.06 | 0.88 +/- 0.07 |
| MOGONET GAT test | 0.87 +/- 0.06 | 0.86 +/- 0.07 | 0.87 +/- 0.06 |


In [65]:
print("MOGONET GAT (mrna, mirna, meth) results:")
print(f"| MOGONET GAT val | {val[:, 0].mean():.2f} +/- {val[:, 0].std():.2f} | {val[:, 1].mean():.2f} +/- {val[:, 1].std():.2f} | {val[:, 2].mean():.2f} +/- {val[:, 2].std():.2f} |")
print(f"| MOGONET GAT test | {test[:, 0].mean():.2f} +/- {test[:, 0].std():.2f} | {test[:, 1].mean():.2f} +/- {test[:, 1].std():.2f} | {test[:, 2].mean():.2f} +/- {test[:, 2].std():.2f} |")

MOGONET GAT (mrna, mirna, meth) results:
| MOGONET GAT val | 0.88 +/- 0.08 | 0.89 +/- 0.07 | 0.88 +/- 0.08 |
| MOGONET GAT test | 0.88 +/- 0.05 | 0.88 +/- 0.05 | 0.89 +/- 0.05 |


In [43]:
print("MOGONET GAT (mrna, mirna, meth) results:")
print(f"| MOGONET GAT val | {val[:, 0].mean():.2f} +/- {val[:, 0].std():.2f} | {val[:, 1].mean():.2f} +/- {val[:, 1].std():.2f} | {val[:, 2].mean():.2f} +/- {val[:, 2].std():.2f} |")
print(f"| MOGONET GAT test | {test[:, 0].mean():.2f} +/- {test[:, 0].std():.2f} | {test[:, 1].mean():.2f} +/- {test[:, 1].std():.2f} | {test[:, 2].mean():.2f} +/- {test[:, 2].std():.2f} |")

MOGONET GAT (mrna, mirna, meth) results:
| MOGONET GAT val | 0.83 +/- 0.05 | 0.83 +/- 0.04 | 0.83 +/- 0.05 |
| MOGONET GAT test | 0.83 +/- 0.10 | 0.84 +/- 0.10 | 0.83 +/- 0.10 |


In [18]:
print("MOGONET GAT (mrna only) results:")
print(f"| MOGONET GAT (mrna only) val | {val[:, 0].mean():.2f} +/- {val[:, 0].std():.2f} | {val[:, 1].mean():.2f} +/- {val[:, 1].std():.2f} | {val[:, 2].mean():.2f} +/- {val[:, 2].std():.2f} |")
print(f"| MOGONET GAT (mrna only) test | {test[:, 0].mean():.2f} +/- {test[:, 0].std():.2f} | {test[:, 1].mean():.2f} +/- {test[:, 1].std():.2f} | {test[:, 2].mean():.2f} +/- {test[:, 2].std():.2f} |")

| MOGONET GAT (mrna only) val | 0.80 +/- 0.06 | 0.81 +/- 0.05 | 0.81 +/- 0.05 |
| MOGONET GAT (mrna only) test | 0.86 +/- 0.04 | 0.84 +/- 0.05 | 0.86 +/- 0.04 |


In [9]:
for mirna_file in os.listdir('BRCA_PROCESSED_DATA/mrmr_selected/mirna'):
    # print(mirna_file)
    mirna = pl.read_csv(f'BRCA_PROCESSED_DATA/mrmr_selected/{mirna_file}')
    
    # mirna_s = mirna.select(['GENE_NAME'] + all_common_sample_names)
    # mirna_s.write_csv(f'MDS_data_preprocessed/mirna/{mirna_file}')

mirna_fold_0.csv
mirna_fold_1.csv
mirna_fold_4.csv
mirna_fold_3.csv
mirna_fold_2.csv


In [25]:
from gnn_experiments.bipartite_eval import bipartite_eval

bipartite_eval(
    input_omics={
        "mrna": mrna,
        # "meth": meth,
        "mirna": mirna,
    },
    n_input_features={
        "mrna": 200,
        # "meth": 200,
        "mirna": 200,
    },
    multipliers={
        "mrna": 1.5,
        # "meth": 1.5,
        "mirna": 1.5,
    },
    # mrmr_selection_params={
    #     "mrmr_path": "BRCA_PROCESSED_DATA/mrmr_selected",
    # },
    input_omics_interactions={
        ("mrna", "mrna") : {
            "gene_interaction" : {
                "interactant_col_1" : "Official Symbol Interactor A",
                "interactant_col_2" : "Official Symbol Interactor B",
                "interactions_file" : "biogrid_preprocessed_data.csv",
            },
            "protein_interaction" : {
                "interactant_col_1" : "gene1",
                "interactant_col_2" : "gene2",
                "interactions_file" : "string_db/ppi.csv",
            },
        },
        # ("meth", "meth") : {
        #     "gene_interaction" : {
        #         "interactant_col_1" : "Official Symbol Interactor A",
        #         "interactant_col_2" : "Official Symbol Interactor B",
        #         "interactions_file" : "biogrid_preprocessed_data.csv",
        #     },
        #     "protein_interaction" : {
        #         "interactant_col_1" : "gene1",
        #         "interactant_col_2" : "gene2",
        #         "interactions_file" : "string_db/ppi.csv",
        #     },
        # },
        # ("mrna", "meth") : {
        #     "gene_interaction" : {
        #         "interactant_col_1" : "Official Symbol Interactor A",
        #         "interactant_col_2" : "Official Symbol Interactor B",
        #         "interactions_file" : "biogrid_preprocessed_data.csv",
        #     },
        #     "protein_interaction" : {
        #         "interactant_col_1" : "gene1",
        #         "interactant_col_2" : "gene2",
        #         "interactions_file" : "string_db/ppi.csv",
        #     },
        # },
        ("mrna", "mirna") : {
            "gene_interaction" : {
                "interactant_col_1" : "gene",
                "interactant_col_2" : "mirna",
                "interactions_file" : "mirna_mrna_interactions_DB.csv",
            },
        },
        # ("meth", "mirna") : {
        #     "gene_interaction" : {
        #         "interactant_col_1" : "gene",
        #         "interactant_col_2" : "mirna",
        #         "interactions_file" : "mirna_mrna_interactions_DB.csv",
        #     },
        # }
    },
    y=y,
    params={
        "epochs": 500,
        "log_interval": 50,
        "save_best_model": False,
        "l1_lambda": 0.0,
        "hidden_channels": [200, 50, 50, 50, 25],
        "heads": 2,
        "dropout": 0,
        "attention_dropout": 0,
        "integrator_type": "attention",
    }
)

dict_keys(['mrna', 'mirna'])
Fold 1 / 5
torch.Size([200]) torch.Size([200])
isolated sample nodes, isolated gene nodes, mean degree: 
tensor(11) tensor(0) tensor(24.6522)
torch.Size([200]) torch.Size([200])
isolated sample nodes, isolated gene nodes, mean degree: 
tensor(2) tensor(0) tensor(25.4369)
Found 312.0 interactions
Found 990.0 interactions
Found 422.0 interactions
HeteroData(
  omics=[2],
  feature_names=[2],
  y=[483],
  train_mask=[386],
  val_mask=[48],
  test_mask=[49],
  num_relations=7,
  mrna={ x=[483, 200] },
  mrna_feature={ x=[200, 200] },
  mirna={ x=[483, 200] },
  mirna_feature={ x=[200, 200] },
  (mrna, diff_exp, mrna_feature)={
    edge_index=[2, 11907],
    edge_attr=[11907, 1],
  },
  (mirna, diff_exp, mirna_feature)={
    edge_index=[2, 12286],
    edge_attr=[12286, 1],
  },
  (mrna_feature, interacts, mrna_feature)={ edge_index=[2, 1298] },
  (mrna_feature, interacts, mirna_feature)={ edge_index=[2, 422] },
  (mrna_feature, rev_diff_exp, mrna)={
    edge_ind

(tensor([[0.6875, 0.7048, 0.6988],
         [0.7708, 0.8077, 0.7714],
         [0.6667, 0.6651, 0.6669],
         [0.7083, 0.7153, 0.7066],
         [0.6667, 0.6458, 0.6744]]),
 tensor([[0.6939, 0.7130, 0.6949],
         [0.8367, 0.8537, 0.8397],
         [0.6939, 0.6202, 0.6426],
         [0.6042, 0.5991, 0.6016],
         [0.7292, 0.7417, 0.7277]]))

In [6]:
val = torch.tensor([[0.9375, 0.9279, 0.9375],
         [0.8542, 0.8671, 0.8519],
         [0.8542, 0.8676, 0.8546],
         [0.7917, 0.7637, 0.7901],
         [0.8333, 0.8551, 0.8358]])
test = torch.tensor([[0.7959, 0.7892, 0.8037],
         [0.8367, 0.8537, 0.8397],
         [0.8776, 0.8746, 0.8783],
         [0.9583, 0.9664, 0.9573],
         [0.8125, 0.8316, 0.8109]])

print("BipartiteGNN (mrna, mirna, meth) results:")
print(f"| BipartiteGNN val | {val[:, 0].mean():.2f} +/- {val[:, 0].std():.2f} | {val[:, 1].mean():.2f} +/- {val[:, 1].std():.2f} | {val[:, 2].mean():.2f} +/- {val[:, 2].std():.2f} |")
print(f"| BipartiteGNN test | {test[:, 0].mean():.2f} +/- {test[:, 0].std():.2f} | {test[:, 1].mean():.2f} +/- {test[:, 1].std():.2f} | {test[:, 2].mean():.2f} +/- {test[:, 2].std():.2f} |")

MOGONET GAT (mrna, mirna, meth) results:
| MOGONET GAT val | 0.85 +/- 0.05 | 0.86 +/- 0.06 | 0.85 +/- 0.05 |
| MOGONET GAT test | 0.86 +/- 0.06 | 0.86 +/- 0.07 | 0.86 +/- 0.06 |


# brca results using all omics


| Method | ACC | F1 | F1 Weighted |
| KNN | 0.79 +/- 0.02 | 0.78 +/- 0.03 | 0.77 +/- 0.03 |
| SVM | 0.83 +/- 0.01 | 0.83 +/- 0.02 | 0.83 +/- 0.01 |
| XGBoost | 0.88 +/- 0.02 | 0.88 +/- 0.03 | 0.88 +/- 0.02 |
| MLP regularization : None | 0.85 +/- 0.03 | 0.85 +/- 0.03 | 0.85 +/- 0.03 |
| MOGONET GAT | 0.86 +/- 0.04 | 0.87 +/- 0.04 | 0.86 +/- 0.04 |

